### Determine the Number of Layers

In [ ]:
from helperfunctions import training_lib as tl
from helperfunctions import intern_constants as ic
from helperfunctions import helper as hfn
from helperfunctions.pretty_print import PrettyPrint as pp
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

In [ ]:
cfg1 = hfn.TrainConfig(config_name="layers_new_wd_09", epochs=10, width_decay=0.9, n_restarts=5, patience=10, min_delta=0,lr=1e-3, weight_decay=1e-2)
cfg2 = hfn.TrainConfig(config_name="layers_new_wd_075", epochs=10, width_decay=0.75, n_restarts=5, patience=10, min_delta=0,lr=1e-3, weight_decay=1e-2)
cfg4 = hfn.TrainConfig(config_name="layers_new_wd_05", epochs=10, width_decay=0.5, n_restarts=5, patience=10, min_delta=0,lr=1e-3, weight_decay=1e-2)

### Determining max. depth of the network

In [ ]:
cfg1.layer_depths = tl.max_depth_for_width_decay(cfg1.base_width, cfg1.width_decay, cfg1.bottleneck_min)
cfg2.layer_depths = tl.max_depth_for_width_decay(cfg2.base_width, cfg2.width_decay, cfg2.bottleneck_min)
# cfg3.layer_depths = tl.max_depth_for_width_decay(cfg3.base_width, cfg3.width_decay, cfg3.bottleneck_min)
cfg4.layer_depths = tl.max_depth_for_width_decay(cfg4.base_width, cfg4.width_decay, cfg4.bottleneck_min)

cfg1.layer_depths = cfg1.layer_depths[14:]
cfg2.layer_depths = cfg2.layer_depths[5:]
cfg4.layer_depths = cfg4.layer_depths[2:]
print(f"cfg1: layer_depths={cfg1.layer_depths}")
print(f"cfg2: layer_depths={cfg2.layer_depths}")
print(f"cfg4: layer_depths={cfg4.layer_depths}")

In [ ]:
cfg_list = [cfg1, 
            cfg2,
            cfg4,
            ]

print(cfg1.layer_depths)


In [ ]:
print(f"val_start_time: {cfg1.val_start_time}\n"
      f"val_end_time: {cfg1.val_end_time}\n"
      f"test_start_time: {cfg1.test_start_time}\n"
      f"test_start_end: {cfg1.test_end_time}\n")

In [ ]:
train_loader, val_loader, test_loader = hfn.build_dataloaders(
                  train_csv_dir=ic.PATH_PC_FILTERING,
                  val_csv_dir=ic.PATH_IMPUTED,
                  test_csv_dir=ic.PATH_IMPUTED,
                  cfg=cfg1
                  )

In [ ]:
scaler = torch.amp.GradScaler() if torch.cuda.is_available() else None
for cfg in cfg_list:

    depths = cfg.layer_depths

    for depth in tqdm(depths):
        
        cfg.depth = int(depth)
        cfg.config_name= f"layers_{depth}"
        
        for i in range(0, cfg.n_restarts):
            # cfg.seed = cfg.seed_list[i]
            cfg.set_seed(cfg.seed_list[i])
            model = tl.Autoencoder(cfg=cfg).to(cfg.device)
            optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
            
            loss_fn = nn.MSELoss(reduction="none")
            es = tl.EarlyStopping(min_delta=cfg.min_delta, patience=cfg.patience)
            
            results_es, _= tl.train_with_early_stopping(
                                                        model,
                                                        train_loader=train_loader,
                                                        val_loader= val_loader,
                                                        optimizer=optimizer,
                                                        config=cfg,
                                                        es=es,
                                                        loss_fn=loss_fn,
                                                        filename_prefix=f"cfg-{cfg.config_name}-layer_depths_{depth}_seed_{cfg.seed}_decay_{cfg.width_decay}",
                                                        scaler=scaler,
                                                        )
            train_loader = hfn.rebuild_grouped_loader(train_loader,
                                                seed=cfg.seed, 
                                                shuffle=True, 
                                                batch_size=cfg.batch_size)
            val_loader = hfn.rebuild_grouped_loader(val_loader, 
                                                seed=cfg.seed,
                                                shuffle=False,
                                                batch_size=cfg.batch_size)
            pp.print_learning_curve(results_es["history"], save_dir=results_es["dir_path"])

In [ ]:
model, (min,max) = tl.get_model_results(ic.PATH_TO_BEST_MODEL_DIR,best_n=1, report_min_max=False)